# Chapter 7: Unsupervised Learning

**Author:** Ruben  
**Date:** September 29, 2026  
**Source:** Practical Statistics for Data Scientists (O'Reilly)

---

## Overview

Unsupervised learning finds patterns without labels:
- Principal Component Analysis (PCA)
- K-Means Clustering
- Hierarchical Clustering
- Dimensionality reduction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
sns.set_style('whitegrid')

## 7.1 Principal Component Analysis (PCA)

### Theory

**PCA:** Dimensionality reduction technique

**Goal:** Find new coordinate system where:
- 1st axis (PC1) captures most variance
- 2nd axis (PC2) captures 2nd most variance (orthogonal to PC1)
- And so on...

**Steps:**
1. Standardize data (mean=0, variance=1)
2. Compute covariance matrix
3. Find eigenvectors (directions) and eigenvalues (variance)
4. Sort by eigenvalues
5. Project data onto top k eigenvectors

**Uses:**
- Dimensionality reduction
- Visualization (reduce to 2D/3D)
- Remove collinearity
- Noise reduction

**Variance explained:** How much information retained by k components

In [ ]:
# PCA example
from sklearn.datasets import load_iris

# Load iris dataset (4 features)
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

print(f'Original data shape: {X.shape}')
print(f'Features: {feature_names}')

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Variance explained
print(f'\nVariance explained by each component:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'PC{i+1}: {var:.3f} ({var*100:.1f}%)')

cumsum = np.cumsum(pca.explained_variance_ratio_)
print(f'\nCumulative variance:')
for i, cum in enumerate(cumsum):
    print(f'First {i+1} PCs: {cum:.3f} ({cum*100:.1f}%)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1), 
           pca.explained_variance_ratio_, edgecolor='black', alpha=0.7)
axes[0].plot(range(1, len(cumsum)+1), cumsum, 'ro-', linewidth=2, label='Cumulative')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Scree Plot')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2D projection
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', 
                         edgecolor='black', alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[1].set_title('Data Projected onto First 2 Principal Components')
plt.colorbar(scatter, ax=axes[1], label='Species')

plt.tight_layout()
plt.show()

print(f'\n✓ First 2 PCs capture {cumsum[1]*100:.1f}% of variance')
print(f'✓ Reduced 4D to 2D while retaining most information')

## 7.2 K-Means Clustering

### Theory

**K-Means:** Partition data into k clusters

**Algorithm:**
1. Choose k random centroids
2. Assign each point to nearest centroid
3. Update centroids (mean of assigned points)
4. Repeat 2-3 until convergence

**Goal:** Minimize within-cluster sum of squares
$$\min \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$$

**Choosing k:**
- Elbow method: plot inertia vs k
- Silhouette score
- Domain knowledge

**Limitations:**
- Must specify k
- Sensitive to initialization
- Assumes spherical clusters
- Sensitive to outliers

In [ ]:
# K-Means clustering example
from sklearn.datasets import make_blobs

# Generate sample data
X_blob, y_true = make_blobs(n_samples=300, centers=4, n_features=2, 
                            cluster_std=0.6, random_state=42)

# Elbow method to find optimal k
inertias = []
K_range = range(1, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_blob)
    inertias.append(kmeans.inertia_)

# Fit with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X_blob)
centers = kmeans.cluster_centers_

print(f'K-Means with k=4:')
print(f'Inertia (within-cluster SS): {kmeans.inertia_:.2f}')
print(f'Number of iterations: {kmeans.n_iter_}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertias, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method for Optimal k')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(4, color='red', linestyle='--', label='Elbow at k=4')
axes[0].legend()

# Clustering result
scatter = axes[1].scatter(X_blob[:, 0], X_blob[:, 1], c=y_pred, cmap='viridis',
                         edgecolor='black', alpha=0.6)
axes[1].scatter(centers[:, 0], centers[:, 1], c='red', s=300, marker='X',
               edgecolor='black', linewidth=2, label='Centroids')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title('K-Means Clustering (k=4)')
axes[1].legend()
plt.colorbar(scatter, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

## 7.3 Hierarchical Clustering

### Theory

**Hierarchical Clustering:** Build tree of clusters

**Agglomerative (bottom-up):**
1. Start: each point is own cluster
2. Merge two closest clusters
3. Repeat until one cluster
4. Cut tree at desired height

**Linkage methods:**
- **Single**: min distance between clusters
- **Complete**: max distance
- **Average**: average distance
- **Ward**: minimize variance

**Dendrogram:** Tree visualization showing merge sequence

**Advantages over K-Means:**
- Don't need to specify k upfront
- Hierarchical structure
- Deterministic (no random initialization)

In [ ]:
# Hierarchical clustering example
# Use smaller dataset for clear dendrogram
X_small = X_blob[:50]

# Compute linkage matrix
linkage_matrix = linkage(X_small, method='ward')

# Fit hierarchical clustering
hc = AgglomerativeClustering(n_clusters=4, linkage='ward')
y_hc = hc.fit_predict(X_small)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Dendrogram
dendrogram(linkage_matrix, ax=axes[0])
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Distance')
axes[0].set_title('Dendrogram (Ward Linkage)')
axes[0].axhline(y=10, color='red', linestyle='--', label='Cut height for 4 clusters')
axes[0].legend()

# Clustering result
scatter = axes[1].scatter(X_small[:, 0], X_small[:, 1], c=y_hc, cmap='viridis',
                         s=100, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title('Hierarchical Clustering Result')
plt.colorbar(scatter, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

print('\n✓ Dendrogram shows merge sequence')
print('✓ Cutting at different heights gives different number of clusters')

## Summary: Unsupervised Learning

### Method Comparison

| Method | Purpose | Pros | Cons |
|--------|---------|------|------|
| PCA | Dimensionality reduction | Fast, interpretable | Linear only |
| K-Means | Clustering | Fast, scalable | Need to specify k |
| Hierarchical | Clustering | No k needed, tree structure | Slow for large data |

### Key Takeaways
- **PCA**: Reduce dimensions while keeping variance
- **K-Means**: Fast clustering, use elbow method for k
- **Hierarchical**: Dendrogram shows cluster structure
- Always standardize features
- Unsupervised = exploratory, not definitive
- Validate with domain knowledge